# WENO TGV Validation — Shock-Turbulence Interaction

Post-processes results from `run_weno_tgv.sh`. The Taylor-Green Vortex is run at three Mach numbers:

- **M = 0.1**: Nearly incompressible baseline. No shocks. All schemes should behave similarly.
- **M = 0.5**: Weak shocklets form during turbulent transition (~t = 5–9). WENO dissipation becomes visible.
- **M = 1.25**: Strong shocklets. C6 is expected to **crash**. WENO should **survive**.

The key diagnostic plots are TKE decay and enstrophy evolution. WENO schemes will show lower enstrophy peaks
(more dissipation) but remain stable where central schemes fail.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, glob, re

# ──── CONFIGURATION ────
RUNDIR = 'weno_tgv_runs'
PLOTDIR = os.path.join(RUNDIR, 'plots')
os.makedirs(PLOTDIR, exist_ok=True)

SCHEMES = ['C6', 'WENO_JS', 'WENO_Z', 'WENO_CU6', 'WENO_CU6M']
MACHS   = [0.1, 0.5, 1.25]
MLABELS = ['M010', 'M050', 'M125']

STYLES = {
    'C6':       dict(color='black',   ls='-',  lw=2.5, label='C6 (baseline)'),
    'WENO_JS':  dict(color='#e41a1c', ls='--', lw=1.5, label='WENO-JS'),
    'WENO_Z':   dict(color='#377eb8', ls='--', lw=1.5, label='WENO-Z'),
    'WENO_CU6': dict(color='#4daf4a', ls='-.', lw=1.5, label='WENO-CU6'),
    'WENO_CU6M':dict(color='#984ea3', ls='-.', lw=1.5, label='WENO-CU6-M'),
}

def load_monitor(scheme, mlabel):
    """Load monitor file: returns (time, tke, enstrophy) arrays or None if crashed."""
    path = os.path.join(RUNDIR, f'{scheme}_{mlabel}', 'Monitor.out')
    if not os.path.exists(path):
        return None
    try:
        data = np.loadtxt(path, comments='#')
        if data.ndim == 1:
            return None  # single line or malformed
        if data.shape[1] < 3:
            return None
        return data[:, 0], data[:, 1], data[:, 2]  # time, tke, enstrophy
    except:
        return None

def load_residuals(scheme, mlabel):
    """Load residuals from log file: returns (time, max_res) arrays or None."""
    path = os.path.join(RUNDIR, f'{scheme}_{mlabel}', 'log.txt')
    if not os.path.exists(path):
        return None
    times, resids = [], []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                try:
                    t = float(parts[0])
                    r = max(abs(float(parts[k])) for k in range(1, 6))
                    times.append(t)
                    resids.append(r)
                except (ValueError, IndexError):
                    continue
    if not times:
        return None
    return np.array(times), np.array(resids)

def run_status(scheme, mlabel):
    """Return status string for a run."""
    logpath = os.path.join(RUNDIR, f'{scheme}_{mlabel}', 'log.txt')
    if not os.path.exists(logpath):
        return 'MISSING'
    with open(logpath) as f:
        text = f.read()
    if 'NaN' in text or 'Infinity' in text:
        return 'DIVERGED'
    if 'Writing output done' in text:
        return 'OK'
    return 'CRASHED'

print(f'Run directory: {os.path.abspath(RUNDIR)}')

## Step 0: Status Matrix — What Survived?

In [ ]:
print(f'{"":>12}', end='')
for s in SCHEMES:
    print(f'{s:>14}', end='')
print()
print('─' * (12 + 14 * len(SCHEMES)))

for mach, ml in zip(MACHS, MLABELS):
    print(f'M={mach:<9}', end='')
    for s in SCHEMES:
        st = run_status(s, ml)
        if st == 'OK':
            print(f'{"✓ OK":>14}', end='')
        elif st == 'DIVERGED':
            print(f'{"✗ DIVERGED":>14}', end='')
        elif st == 'CRASHED':
            print(f'{"✗ CRASHED":>14}', end='')
        else:
            print(f'{"— MISSING":>14}', end='')
    print()

print()
# Key result check
c6_m125 = run_status('C6', 'M125')
weno_m125 = [run_status(s, 'M125') for s in SCHEMES if s != 'C6']
if c6_m125 in ('DIVERGED', 'CRASHED') and all(s == 'OK' for s in weno_m125):
    print('\033[92m★ KEY RESULT: C6 crashed at M=1.25, all WENO schemes survived!\033[0m')
    print('  → WENO shock-capturing works correctly for shock-turbulence interaction.')
elif c6_m125 == 'OK':
    print('C6 survived at M=1.25 — the filter is strong enough to handle the shocklets.')
    print('Compare enstrophy peaks: WENO should show more dissipation.')

## Plot 1: Kinetic Energy Decay

The volume-averaged kinetic energy $E_k = \frac{1}{2}\langle u_i u_i \rangle$ should decay monotonically after transition.
More dissipative schemes (JS) cause faster decay. Less dissipative (CU6-M) track closer to C6.

In [ ]:
fig, axes = plt.subplots(1, len(MACHS), figsize=(6*len(MACHS), 5), sharey=False)
if len(MACHS) == 1:
    axes = [axes]

for ax, mach, ml in zip(axes, MACHS, MLABELS):
    for scheme in SCHEMES:
        result = load_monitor(scheme, ml)
        if result is None:
            continue
        t, tke, _ = result
        s = STYLES[scheme]
        ax.plot(t, tke, color=s['color'], ls=s['ls'], lw=s['lw'], label=s['label'])
    
    ax.set_xlabel('t / t_c', fontsize=12)
    ax.set_ylabel('Kinetic Energy, $E_k$', fontsize=12)
    ax.set_title(f'M = {mach}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, None)

fig.suptitle('TGV: Kinetic Energy Decay (Re = 1600)', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'TGV_kinetic_energy.pdf'), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(PLOTDIR, 'TGV_kinetic_energy.png'), dpi=200, bbox_inches='tight')
plt.show()

## Plot 2: Enstrophy Evolution

Enstrophy $\Omega = \langle \omega_i \omega_i \rangle$ peaks at $t \approx 9$ for Re = 1600.
This is the most sensitive diagnostic — it measures small-scale vorticity generation during transition.
More dissipative schemes underpredict the peak.

In [ ]:
fig, axes = plt.subplots(1, len(MACHS), figsize=(6*len(MACHS), 5), sharey=False)
if len(MACHS) == 1:
    axes = [axes]

for ax, mach, ml in zip(axes, MACHS, MLABELS):
    for scheme in SCHEMES:
        result = load_monitor(scheme, ml)
        if result is None:
            continue
        t, _, enst = result
        s = STYLES[scheme]
        ax.plot(t, enst, color=s['color'], ls=s['ls'], lw=s['lw'], label=s['label'])
    
    ax.set_xlabel('t / t_c', fontsize=12)
    ax.set_ylabel('Enstrophy, $\\Omega$', fontsize=12)
    ax.set_title(f'M = {mach}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, None)

fig.suptitle('TGV: Enstrophy Evolution (Re = 1600)', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'TGV_enstrophy.pdf'), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(PLOTDIR, 'TGV_enstrophy.png'), dpi=200, bbox_inches='tight')
plt.show()

## Plot 3: Kinetic Energy Dissipation Rate

The dissipation rate $\epsilon = -dE_k/dt$ is computed via central differences of the TKE curve.
This is the quantity most often compared in TGV benchmarks (e.g. DeBonis 2013, Bull & Jameson 2015).
The peak in dissipation rate corresponds to the most intense period of turbulent transition.

In [ ]:
fig, axes = plt.subplots(1, len(MACHS), figsize=(6*len(MACHS), 5), sharey=False)
if len(MACHS) == 1:
    axes = [axes]

for ax, mach, ml in zip(axes, MACHS, MLABELS):
    for scheme in SCHEMES:
        result = load_monitor(scheme, ml)
        if result is None:
            continue
        t, tke, _ = result
        # Central differences for -dE_k/dt, smoothed
        if len(t) < 5:
            continue
        dt_arr = np.diff(t)
        dtke = np.diff(tke)
        diss_rate = -dtke / dt_arr
        t_mid = 0.5 * (t[:-1] + t[1:])
        # Light smoothing (moving average, window=5)
        if len(diss_rate) > 10:
            kernel = np.ones(5) / 5
            diss_rate = np.convolve(diss_rate, kernel, mode='same')
        
        s = STYLES[scheme]
        ax.plot(t_mid, diss_rate, color=s['color'], ls=s['ls'], lw=s['lw'], label=s['label'])
    
    ax.set_xlabel('t / t_c', fontsize=12)
    ax.set_ylabel('$-dE_k/dt$', fontsize=12)
    ax.set_title(f'M = {mach}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, None)
    ax.set_ylim(bottom=0)

fig.suptitle('TGV: KE Dissipation Rate (Re = 1600)', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'TGV_dissipation_rate.pdf'), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(PLOTDIR, 'TGV_dissipation_rate.png'), dpi=200, bbox_inches='tight')
plt.show()

## Plot 4: Residual History

Max residual (across all 5 conserved variables) vs time.
A diverging run shows exponential residual growth before crashing.

In [ ]:
fig, axes = plt.subplots(1, len(MACHS), figsize=(6*len(MACHS), 5), sharey=False)
if len(MACHS) == 1:
    axes = [axes]

for ax, mach, ml in zip(axes, MACHS, MLABELS):
    for scheme in SCHEMES:
        res = load_residuals(scheme, ml)
        if res is None:
            continue
        t, r = res
        # Clip for log plot
        r = np.clip(r, 1e-20, None)
        s = STYLES[scheme]
        ax.semilogy(t, r, color=s['color'], ls=s['ls'], lw=s['lw'], label=s['label'], alpha=0.8)
    
    ax.set_xlabel('t / t_c', fontsize=12)
    ax.set_ylabel('Max Residual', fontsize=12)
    ax.set_title(f'M = {mach}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, which='both', alpha=0.3)
    ax.set_xlim(0, None)

fig.suptitle('TGV: Residual History (Re = 1600)', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'TGV_residuals.pdf'), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(PLOTDIR, 'TGV_residuals.png'), dpi=200, bbox_inches='tight')
plt.show()

## Plot 5: Enstrophy Peak Comparison

Bar chart comparing the maximum enstrophy reached by each scheme at each Mach number.
Missing bars = crashed runs (the scheme couldn't handle the shocklets).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = np.arange(len(MACHS))
bar_width = 0.15
offsets = np.arange(len(SCHEMES)) - len(SCHEMES)/2 + 0.5

for idx, scheme in enumerate(SCHEMES):
    peaks = []
    for ml in MLABELS:
        result = load_monitor(scheme, ml)
        if result is None:
            peaks.append(0)
        else:
            _, _, enst = result
            peaks.append(np.max(enst))
    
    s = STYLES[scheme]
    bars = ax.bar(x_pos + offsets[idx]*bar_width, peaks, bar_width,
                  color=s['color'], label=s['label'], alpha=0.85, edgecolor='white')
    # Mark crashed runs
    for i, (p, ml) in enumerate(zip(peaks, MLABELS)):
        if p == 0:
            ax.annotate('✗', xy=(x_pos[i] + offsets[idx]*bar_width, 0.1),
                       ha='center', fontsize=12, color='red', fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels([f'M = {m}' for m in MACHS], fontsize=12)
ax.set_ylabel('Peak Enstrophy', fontsize=13)
ax.set_title('TGV: Enstrophy Peak Comparison (Re = 1600)', fontsize=14)
ax.legend(fontsize=9, ncol=3, loc='upper right')
ax.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, 'TGV_enstrophy_peaks.pdf'), dpi=150)
fig.savefig(os.path.join(PLOTDIR, 'TGV_enstrophy_peaks.png'), dpi=200)
plt.show()

## Summary Table

In [ ]:
print('=' * 85)
print('  WENO TGV SUMMARY')
print('=' * 85)
print()

print('  Enstrophy peaks:')
print(f'{"":>12}', end='')
for s in SCHEMES:
    print(f'{s:>14}', end='')
print()
print('  ' + '─' * (10 + 14*len(SCHEMES)))

for mach, ml in zip(MACHS, MLABELS):
    print(f'  M={mach:<7}', end='')
    for s in SCHEMES:
        result = load_monitor(s, ml)
        if result is None:
            st = run_status(s, ml)
            print(f'{st:>14}', end='')
        else:
            _, _, enst = result
            print(f'{np.max(enst):>14.3f}', end='')
    print()

print()
print('  Time of enstrophy peak:')
print(f'{"":>12}', end='')
for s in SCHEMES:
    print(f'{s:>14}', end='')
print()
print('  ' + '─' * (10 + 14*len(SCHEMES)))

for mach, ml in zip(MACHS, MLABELS):
    print(f'  M={mach:<7}', end='')
    for s in SCHEMES:
        result = load_monitor(s, ml)
        if result is None:
            print(f'{"—":>14}', end='')
        else:
            t, _, enst = result
            peak_idx = np.argmax(enst)
            print(f'{t[peak_idx]:>14.2f}', end='')
    print()

print()
print('=' * 85)
print('  Interpretation:')
print('  • Higher enstrophy peak = less numerical dissipation = better resolution of small scales')
print('  • JS should have the lowest peaks (most dissipative WENO variant)')
print('  • CU6-M should be closest to C6 (least dissipative WENO variant)')
print('  • At M=1.25, a crashed C6 and surviving WENO proves the shock-capturing works')
print('  • At M=0.5, all should survive but WENO peaks will be slightly lower than C6')
print('=' * 85)

## What to Look For

**M = 0.1 (no shocks):**
All schemes should produce nearly identical TKE and enstrophy curves (the flow is smooth).
Any differences show the baseline dissipation of each scheme.
WENO-JS will have slightly lower enstrophy peak than C6.

**M = 0.5 (weak shocklets):**
Shocklets form during transition (t ≈ 5–9). WENO's built-in dissipation captures them cleanly.
C6 may survive (the filter provides some shock-handling) but might show slight oscillations.
WENO peaks will be lower than C6 — more dissipation is the price of shock-capturing.

**M = 1.25 (strong shocklets):**
The critical test. C6 with F10 filter should crash (Gibbs oscillations → negative pressure → NaN).
All WENO variants should survive. Among them:
- JS: lowest enstrophy peak (most dissipative), most robust
- Z: slightly better, still very robust
- CU6: good balance of accuracy and stability
- CU6-M: highest peak (closest to DNS), but the C=1000/q=4 parameters
  push it closer to the linear scheme, which might cause mild oscillations

**References:**
- DeBonis (2013), AIAA-2013-0382: compressible TGV benchmark at M=0.1
- Bull & Jameson (2015): compressible TGV at M=0.1, 0.5, 1.25
- Lusher & Sandham (2021): compressible TGV DNS at various Mach numbers